# 06 — Exploratory data analysis and maps

This notebook supports the core research question:

> **How has artificial nighttime brightness evolved across European cities over the past decade, and which urban characteristics are associated with higher levels and faster growth of light pollution?**

The analysis panel merges **VIIRS** nighttime lights (brightness, pixels), **OpenStreetMap**-derived urban form (roads, buildings, land use, green space), and **Eurostat** socioeconomic indicators (population, GDP per capita, density). Formal time series, regression, and clustering live in `07_time_series.ipynb`, `08_regression.ipynb`, and `09_clustering.ipynb`; here we **profile the panel**, explore **levels and growth** descriptively, and **map** spatial patterns.


## Objectives
1. Profile the panel (coverage, missingness, static vs time-varying fields).
2. Describe how **brightness levels** and **year-over-year / decade growth** vary over time and across cities.
3. Explore **bivariate patterns** between brightness (and growth) and urban / socioeconomic variables.
4. Map **levels**, **decade change**, and **estimated growth rates** across Europe and save figures for the report.


In [ ]:
import warnings
from pathlib import Path

import folium
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from branca.colormap import linear
from IPython.display import display

warnings.filterwarnings("ignore", category=UserWarning, module="matplotlib")

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 11

FIG_DIR = Path("../outputs/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
file_path = Path("../data/processed/analysis_panel.csv")
df = pd.read_csv(file_path)

df.columns = df.columns.str.strip()
if "year" in df.columns:
    df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")

y0, y1 = int(df["year"].min()), int(df["year"].max())
print(f"Rows: {df.shape[0]:,}  |  Columns: {df.shape[1]}  |  Cities: {df['city'].nunique()}")
print(f"Years: {y0}–{y1}  |  Countries: {df['country'].nunique()}")

display(df.head(3))


## 1. Dataset profile and quality checks


In [ ]:
city_year = df.pivot_table(
    index="city", columns="year", values="mean_brightness", aggfunc="count"
)
plt.figure(figsize=(14, max(4, df["city"].nunique() * 0.18)))
sns.heatmap(city_year.notna(), cbar=False, cmap="YlGnBu", linewidths=0)
plt.title("Observed city–year cells (non-missing mean brightness)")
plt.xlabel("Year")
plt.ylabel("City")
plt.tight_layout()
plt.savefig(FIG_DIR / "eda_city_year_coverage.png", dpi=150, bbox_inches="tight")
plt.show()

coverage = (
    df.groupby("year", dropna=False)["city"]
    .nunique()
    .reset_index(name="n_cities")
    .sort_values("year")
)
print("Cities per year:")
display(coverage)

missing = (
    df.isna()
    .mean()
    .sort_values(ascending=False)
    .rename("missing_share")
    .reset_index()
    .rename(columns={"index": "column"})
)
display(missing.head(15))

plt.figure(figsize=(10, 5))
sns.barplot(data=missing.head(12), x="missing_share", y="column", hue="column", palette="viridis", legend=False)
plt.title("Top 12 columns by missing share")
plt.xlabel("Missing share")
plt.ylabel("")
plt.xlim(0, 1)
plt.tight_layout()
plt.savefig(FIG_DIR / "eda_missingness_top12.png", dpi=150, bbox_inches="tight")
plt.show()

key_num = [
    "mean_brightness",
    "median_brightness",
    "pixel_count",
    "population",
    "gdp_per_capita_eur",
    "built_up_fraction",
    "green_fraction",
    "road_density_km_per_km2",
    "brightness_cagr",
]
key_num = [c for c in key_num if c in df.columns]
display(df[key_num].describe(percentiles=[0.05, 0.5, 0.95]).T)


## 2. Temporal patterns in night light brightness


In [ ]:
annual = (
    df.groupby("year", as_index=False)
    .agg(mean_b=("mean_brightness", "mean"), median_b=("median_brightness", "median"))
    .sort_values("year")
)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(annual["year"], annual["mean_b"], marker="o", label="Cross-city mean", color="#1f77b4")
ax.plot(annual["year"], annual["median_b"], marker="s", label="Cross-city median of city medians", color="#ff7f0e")
ax.set_title("Aggregate brightness trajectory (interpret alongside VIIRS product notes)")
ax.set_xlabel("Year")
ax.set_ylabel("Brightness (VIIRS units)")
ax.legend()
ax.grid(True, alpha=0.3, linestyle="--")
plt.tight_layout()
plt.savefig(FIG_DIR / "eda_aggregate_brightness_trend.png", dpi=150, bbox_inches="tight")
plt.show()

latest_year = int(df["year"].dropna().max())
latest = df[df["year"] == latest_year].copy()
print(f"Top 8 brightest cities in {latest_year}:")
display(latest.nlargest(8, "mean_brightness")[["city", "country", "mean_brightness"]])

rng = np.random.default_rng(42)
cities = df["city"].dropna().unique().tolist()
sample = sorted(rng.choice(cities, size=min(12, len(cities)), replace=False))
sub = df[df["city"].isin(sample)].sort_values(["city", "year"])

fig, ax = plt.subplots(figsize=(12, 6))
for city, g in sub.groupby("city"):
    ax.plot(g["year"], g["mean_brightness"], alpha=0.75, linewidth=1.5, label=city)
ax.set_title("Mean brightness trajectories (random sample of cities)")
ax.set_xlabel("Year")
ax.set_ylabel("Mean brightness")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.savefig(FIG_DIR / "eda_brightness_spaghetti_sample.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(data=df, x="year", y="mean_brightness", color="#9ecae1", fliersize=2)
plt.title("Distribution of mean brightness by year")
plt.xlabel("Year")
plt.ylabel("Mean brightness")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(FIG_DIR / "eda_brightness_boxplot_by_year.png", dpi=150, bbox_inches="tight")
plt.show()

if "brightness_yoy" in df.columns:
    print("Brightness YoY (pooled across city-years):")
    display(df["brightness_yoy"].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]).to_frame("value"))

if "brightness_cagr" in df.columns:
    cg = df.drop_duplicates("city")[["city", "country", "brightness_cagr"]].dropna(subset=["brightness_cagr"])
    cg = cg.sort_values("brightness_cagr")
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    sns.histplot(cg["brightness_cagr"], kde=True, ax=axes[0], color="#9467bd")
    axes[0].set_title("Distribution of city brightness CAGR (as defined in merge panel)")
    axes[0].set_xlabel("CAGR")
    plot_df = pd.concat(
        [
            cg.head(10).assign(group="Lowest 10"),
            cg.tail(10).iloc[::-1].assign(group="Highest 10"),
        ]
    )
    sns.barplot(data=plot_df, y="city", x="brightness_cagr", hue="group", ax=axes[1])
    axes[1].set_title("Cities with lowest vs highest CAGR")
    axes[1].axvline(0, color="k", linewidth=0.8, linestyle="--")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "eda_brightness_cagr_distribution.png", dpi=150, bbox_inches="tight")
    plt.show()


## 3. Cross-variable EDA (brightness vs urban and socioeconomic factors)


In [ ]:
corr_cols = [
    "mean_brightness",
    "population",
    "gdp_per_capita_eur",
    "population_density",
    "road_density_km_per_km2",
    "motorway_length_km",
    "street_lamp_count",
    "building_count",
    "built_up_fraction",
    "green_fraction",
    "landuse_commercial_km2",
    "landuse_industrial_km2",
    "park_area_km2",
    "brightness_per_capita",
    "brightness_cagr",
]
corr_cols = [c for c in corr_cols if c in df.columns]
corr_matrix = df[corr_cols].corr(numeric_only=True)

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, cmap="coolwarm", center=0, annot=False, square=True)
plt.title("Correlation matrix (pooled panel; use regression notebook for inference)")
plt.tight_layout()
plt.savefig(FIG_DIR / "eda_correlation_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

ly = int(df["year"].dropna().max())
last = df[df["year"] == ly].copy()

scatter_specs = [
    ("population", "mean_brightness", "Population"),
    ("gdp_per_capita_eur", "mean_brightness", "GDP per capita (EUR)"),
    ("built_up_fraction", "mean_brightness", "Built-up fraction (OSM)"),
    ("road_density_km_per_km2", "mean_brightness", "Road density (km/km²)"),
    ("green_fraction", "mean_brightness", "Green fraction (OSM)"),
    ("street_lamp_count", "mean_brightness", "Street lamp count (OSM)"),
]
scatter_specs = [(x, y, xl) for x, y, xl in scatter_specs if x in last.columns and y in last.columns]

nc = 3
nr = int(np.ceil(len(scatter_specs) / nc))
fig, axes = plt.subplots(nr, nc, figsize=(5 * nc, 4 * nr))
axes = np.atleast_1d(axes).ravel()
for ax, (xcol, ycol, xlab) in zip(axes, scatter_specs):
    sns.regplot(data=last, x=xcol, y=ycol, scatter_kws={"alpha": 0.55}, line_kws={"color": "crimson"}, ax=ax)
    ax.set_title(f"Mean brightness vs {xlab} ({ly})")
    ax.set_ylabel("Mean brightness")
    ax.grid(True, alpha=0.25, linestyle="--")
for j in range(len(scatter_specs), len(axes)):
    axes[j].set_visible(False)
plt.tight_layout()
plt.savefig(FIG_DIR / "eda_scattergrid_brightness_latest.png", dpi=150, bbox_inches="tight")
plt.show()

if "brightness_cagr" in last.columns:
    city_growth = last.dropna(subset=["brightness_cagr"]).copy()
    triples = [
        ("built_up_fraction", "Built-up fraction"),
        ("road_density_km_per_km2", "Road density"),
        ("green_fraction", "Green fraction"),
    ]
    triples = [(a, b) for a, b in triples if a in city_growth.columns]
    if triples:
        fig, axes = plt.subplots(1, len(triples), figsize=(5 * len(triples), 4))
        axes = np.atleast_1d(axes).ravel()
        for ax, (xc, xlab) in zip(axes, triples):
            sns.regplot(
                data=city_growth,
                x=xc,
                y="brightness_cagr",
                scatter_kws={"alpha": 0.65},
                line_kws={"color": "darkred"},
                ax=ax,
            )
            ax.axhline(0, color="k", linewidth=0.7, linestyle="--")
            ax.set_title(f"CAGR vs {xlab}")
            ax.grid(True, alpha=0.25, linestyle="--")
        plt.suptitle("Brightness growth vs urban indicators (descriptive)", y=1.05)
        plt.tight_layout()
        plt.savefig(FIG_DIR / "eda_scatter_cagr_vs_urban.png", dpi=150, bbox_inches="tight")
        plt.show()


In [ ]:
# Decade change in mean brightness (first to last year in panel)
y_first = int(df["year"].dropna().min())
ly = int(df["year"].dropna().max())
b0 = df[df["year"] == y_first][["city", "mean_brightness"]].rename(columns={"mean_brightness": "brightness_first"})
b1 = df[df["year"] == ly][["city", "mean_brightness"]].rename(columns={"mean_brightness": "brightness_last"})
delta = b0.merge(b1, on="city", how="inner")
delta["delta_brightness"] = delta["brightness_last"] - delta["brightness_first"]
delta = delta.sort_values("delta_brightness")

fig, ax = plt.subplots(figsize=(10, 8))
colors = np.where(delta["delta_brightness"] >= 0, "#d62728", "#2ca02c")
ax.barh(delta["city"], delta["delta_brightness"], color=colors, alpha=0.85)
ax.axvline(0, color="k", linewidth=0.8)
ax.set_xlabel(f"Change in mean brightness ({y_first} → {ly})")
ax.set_title("City ranking by decade change in VIIRS mean brightness")
plt.tight_layout()
plt.savefig(FIG_DIR / "eda_delta_brightness_ranking.png", dpi=150, bbox_inches="tight")
plt.show()


## 4. Geospatial analysis


In [ ]:
ly = int(df["year"].dropna().max())
y_first = int(df["year"].dropna().min())

geo_df = df[df["year"] == ly].copy()
geo_df = geo_df.dropna(subset=["lat", "lon", "mean_brightness"])

b0 = df[df["year"] == y_first][["city", "mean_brightness"]].rename(columns={"mean_brightness": "brightness_first"})
delta = b0.merge(
    geo_df[["city", "mean_brightness"]].rename(columns={"mean_brightness": "brightness_last"}),
    on="city",
    how="right",
)
delta["delta_brightness"] = delta["brightness_last"] - delta["brightness_first"]
geo_df = geo_df.merge(delta[["city", "delta_brightness"]], on="city", how="left")

gdf = gpd.GeoDataFrame(
    geo_df,
    geometry=gpd.points_from_xy(geo_df["lon"], geo_df["lat"]),
    crs="EPSG:4326",
)

print(f"Map snapshot: {ly}, n={len(gdf)} cities")
display(gdf[["city", "country", "mean_brightness", "delta_brightness", "brightness_cagr"]].head())


In [ ]:
# Static maps: level vs decade change (Europe extent, no external country file)
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, col, title, cmap in [
    (axes[0], "mean_brightness", f"Mean brightness ({ly})", "plasma"),
    (axes[1], "delta_brightness", f"Change in mean brightness ({y_first} → {ly})", "RdBu_r"),
]:
    gdf.plot(
        ax=ax,
        column=col,
        cmap=cmap,
        legend=True,
        markersize=70,
        alpha=0.92,
        edgecolor="black",
        linewidth=0.35,
    )
    ax.set_xlim(-12, 32)
    ax.set_ylim(36, 72)
    ax.set_aspect("equal")
    ax.set_facecolor("#e8eef2")
    ax.grid(True, alpha=0.35, linestyle=":")
    ax.set_title(title)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")

plt.tight_layout()
plt.savefig(FIG_DIR / "eda_map_europe_level_and_delta.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Optional: Web Mercator basemap (requires network tile fetch)
try:
    import contextily as ctx

    g3857 = gdf.to_crs(3857)
    fig, ax = plt.subplots(figsize=(12, 10))
    g3857.plot(ax=ax, column="mean_brightness", cmap="magma", legend=True, alpha=0.85, markersize=60, edgecolor="white", linewidth=0.4)
    ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron)
    ax.set_title(f"Mean brightness with basemap ({ly})")
    ax.set_axis_off()
    plt.tight_layout()
    plt.savefig(FIG_DIR / "eda_map_basemap_mean_brightness.png", dpi=150, bbox_inches="tight")
    plt.show()
except Exception as e:
    print("Skipping contextily basemap:", e)


In [ ]:
center_lat = float(gdf["lat"].mean())
center_lon = float(gdf["lon"].mean())

m = folium.Map(location=[center_lat, center_lon], zoom_start=4, tiles="cartodbpositron")

min_b, max_b = float(gdf["mean_brightness"].min()), float(gdf["mean_brightness"].max())
colormap = linear.viridis.scale(min_b, max_b)
colormap.caption = f"Mean brightness ({ly})"

for _, row in gdf.iterrows():
    cagr_txt = f"{row['brightness_cagr']:.4f}" if pd.notna(row.get("brightness_cagr")) else "N/A"
    dlt = row.get("delta_brightness")
    dlt_txt = f"{dlt:.2f}" if pd.notna(dlt) else "N/A"
    popup_text = (
        f"<b>{row['city']}, {row['country']}</b><br>"
        f"Year: {ly}<br>"
        f"Mean brightness: {row['mean_brightness']:.2f}<br>"
        f"Δ brightness ({y_first}→{ly}): {dlt_txt}<br>"
        f"CAGR (panel): {cagr_txt}<br>"
        f"Population: {row['population'] if pd.notna(row.get('population')) else 'N/A'}<br>"
        f"GDP/cap (EUR): {row['gdp_per_capita_eur'] if pd.notna(row.get('gdp_per_capita_eur')) else 'N/A'}"
    )
    folium.CircleMarker(
        location=[float(row["lat"]), float(row["lon"])],
        radius=6,
        color=colormap(float(row["mean_brightness"])),
        fill=True,
        fill_opacity=0.85,
        popup=folium.Popup(popup_text, max_width=320),
    ).add_to(m)

colormap.add_to(m)
m


## 5. Key takeaways (fill in after a full run)

Use this checklist to connect figures to the research question (one or two sentences each):

- **Aggregate trend:** Do cross-city mean and median brightness move together? Any single-year jumps that might reflect VIIRS or processing changes rather than cities alone?
- **Heterogeneity:** From the spaghetti plot and CAGR bars, do most cities move in the same direction, or is the sample polarized?
- **Levels vs growth:** Does the correlation pattern suggest that *denser* or *more built-up* places are brighter *today*, and do the same variables align with *faster growth* in light (CAGR), or not?
- **Space:** From the Europe maps, where are the brightest cores and the largest positive/negative decade changes? Do those patterns match priors (e.g. capital vs non-capital, southern vs northern Europe)?
- **Next steps:** Which hypotheses will you take to `08_regression.ipynb` (controls, fixed effects, nonlinearity)?

Figures saved under `outputs/figures/` with prefix `eda_` can be dropped into the report or slides.
